# Headless Fine-Tuning: 'batchcode' (Qwen2.5-Coder-1.5B-Instruct)
### DistillLab Dataset Distillation Pipeline
- **Base Architecture**: `Qwen/Qwen2.5-Coder-1.5B-Instruct`
- **Framework**: Unsloth 4-bit QLoRA
- **Quantized Export**: GGUF (`q4_k_m`) for direct execution in Ollama

In [ ]:
# 1. Install Unsloth and optimized dependencies
!pip install --quiet "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --quiet --no-deps trl peft accelerate bitsandbytes

In [ ]:
# 2. Sanitize and inspect dataset
dataset_file = "datacs_programming_dataset.jsonl"
with open(dataset_file, "r", encoding="utf-8") as f:
    content = f.read()

cleaned = content.replace("#template", "template").replace("#requires", "requires").replace("#concept", "concept")
with open(dataset_file, "w", encoding="utf-8") as f:
    f.write(cleaned)
print("Sanitization verified: 0 invalid preprocessor directives.")

In [ ]:
# 3. Load Qwen 2.5 Coder in 4-bit
from unsloth import FastLanguageModel

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-Coder-1.5B-Instruct",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

# Attach LoRA adapters to Attention + MLP layers
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [ ]:
# 4. Format dataset with ChatML
from datasets import load_dataset

def format_chatml(batch):
    texts = [tokenizer.apply_chat_template(convo, tokenize=False) for convo in batch["messages"]]
    return {"text": texts}

dataset = load_dataset("json", data_files=dataset_file, split="train")
dataset = dataset.map(format_chatml, batched=True)
print(f"Loaded {len(dataset)} verified samples.")

In [ ]:
# 5. Train with SFTTrainer
import torch
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=True,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        output_dir="./outputs_batchcode",
        seed=3407,
    ),
)

trainer.train()

In [ ]:
# 6. Direct Quantized GGUF Export for Ollama ('batchcode')
model.save_pretrained_gguf("batchcode", tokenizer, quantization_method="q4_k_m")
print("GGUF quantization complete in ./batchcode/")

# Create Ollama Modelfile
modelfile = '''FROM ./unsloth.Q4_K_M.gguf

TEMPLATE """{{ if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}{{ if .Prompt }}<|im_start|>user
{{ .Prompt }}<|im_end|>
{{ end }}<|im_start|>assistant
{{ .Response }}<|im_end|>
"""

PARAMETER stop "<|im_start|>"
PARAMETER stop "<|im_end|>"
PARAMETER temperature 0.6
PARAMETER top_p 0.95
PARAMETER repeat_penalty 1.1
'''

with open("batchcode/Modelfile", "w", encoding="utf-8") as f:
    f.write(modelfile)
print("Ollama Modelfile generated in batchcode/Modelfile")

In [ ]:
# 7. Download quantized batchcode artifact
from google.colab import files
!zip -r batchcode_quantized.zip batchcode/
files.download('batchcode_quantized.zip')